# Agent 7 — Visual RAG

> Answer arbitrary questions over a video library by combining cheap
> retrieval (semantic visual + semantic audio) with expensive VLM
> verification per merged candidate. Every claim links to a specific
> timestamp in a specific video.

## What's different from text RAG

Text RAG retrieves documents and reasons over them. Visual RAG
retrieves *video time-ranges* and reasons over them. The retrieval
layer is two-channel (visual + audio), and overlapping hits across
channels are stronger evidence than either alone.

## Endpoints exercised

| Step | Endpoint |
|---|---|
| Retrieve (visual) | `POST /search` BY_CLIP |
| Retrieve (audio) | `POST /search` BY_AUDIO |
| Reason | `POST /vu/chat/completions` per candidate |


## Setup

You need:

1. A Memories.ai API key (`sk-mavi-...`) — get one at the [Memories.ai console](https://api-platform.memories.ai/stripe).
2. Python 3.10+ and the `requests` library (`pip install requests`).

Set the key as an environment variable before launching Jupyter, or paste it
inline in the cell below. The same key works across Visual Search, Visual
Intelligence, and Visual Agents — no separate auth per product.


In [ ]:
import os, json, time, requests

# ────────────────────────────────────────────────────────────────────────────
# Auth: the Memories.ai key is a single token used across every product.
# Pass it as the literal `Authorization` header value — no `Bearer` prefix.
# ────────────────────────────────────────────────────────────────────────────
API_KEY = os.environ.get("MEMORIES_API_KEY") or "sk-mavi-..."  # ← paste here if not using env
HEADERS = {"Authorization": API_KEY}

# Hosts: Visual Search and Visual Intelligence live on different domains.
VS_HOST  = "https://api.memories.ai/serve/api/v1"            # Visual Search
VLM_HOST = "https://mavi-backend.memories.ai/serve/api/v2"   # Visual Intelligence (VLM + Visual Agents)

# Default VLM model used for verification / reasoning. Other options include
# `qwen:qwen2.5-vl-72b-instruct`, `nova:amazon.nova-lite-v1:0`, etc. — see
# Memories.ai docs for the full list and per-token pricing.
VLM_MODEL = "gemini:gemini-2.5-flash"


In [ ]:
def search(query, *, video_nos=None, unique_id="default", top_k=10,
           filtering_level="medium", search_type="BY_CLIP",
           datetime_taken=None, tag=None, camera_tag=None,
           latitude=None, longitude=None, max_retries=3):
    """Visual Search — POST /search.

    Returns the list of {videoNo, startTime, endTime, score, ...} matches.

    Filter parameters (all optional, combinable):
      • video_nos       restrict to specific videos (up to 100 ids)
      • datetime_taken  filter to videos captured at-or-after this timestamp
      • tag             filter to videos carrying this user-defined tag
      • camera_tag      filter to videos shot on this camera_model
      • latitude/longitude  GPS proximity filter (must be paired)

    Notes on the envelope: Visual Search responses are wrapped in
    {code, msg, data, success, failed}. code="0000" means OK; the actual
    hit list is in `data`. code="0001" ("network abnormal") is transient
    and we retry it.
    """
    body = {
        "search_param": query,
        "search_type": search_type,                 # "BY_CLIP" or "BY_AUDIO"
        "unique_id": unique_id,                      # namespace within your account
        "top_k": top_k,
        "filtering_level": filtering_level,          # "low"|"medium"|"high"
    }
    if video_nos:        body["video_nos"]     = list(video_nos)
    if datetime_taken:   body["datetime_taken"] = datetime_taken
    if tag:              body["tag"]            = tag
    if camera_tag:       body["camera_tag"]     = camera_tag
    if latitude is not None and longitude is not None:
        body["latitude"]  = latitude
        body["longitude"] = longitude

    for attempt in range(max_retries):
        r = requests.post(f"{VS_HOST}/search", headers=HEADERS, json=body, timeout=60)
        r.raise_for_status()
        envelope = r.json()
        code = envelope.get("code")
        if code == "0000":
            return envelope.get("data") or []
        if code == "0001" and attempt < max_retries - 1:
            # Transient. Backoff and retry.
            time.sleep(0.4 * (2 ** attempt))
            continue
        raise RuntimeError(f"/search failed: code={code} msg={envelope.get('msg')!r}")
    return []


In [ ]:
def vlm_complete(prompt, *, video_url=None, image_url=None, system=None,
                 model=VLM_MODEL, response_json=True, temperature=0.2,
                 max_tokens=1024):
    """Visual Intelligence — POST /vu/chat/completions.

    Calls a Video Language Model (Gemini by default) with text + an
    optional media reference. Returns the assistant's text reply, with
    markdown ```json fences stripped if `response_json=True`.

    Notes on the wire format:
      • `content` MUST be an array even for text-only prompts. A bare
        string is rejected with "Model input cannot be empty".
      • Gemini's response lives in choices[0]["text"]. Other providers
        (Qwen, Nova) use choices[0]["message"]["content"]. We accept both.
      • The endpoint can return HTTP 200 with status="errored" — surface
        that as a typed exception rather than letting JSON parsing fail.
    """
    content = [{"type": "text", "text": prompt}]
    if video_url:
        content.append({"type": "input_file", "file_uri": video_url, "mime_type": "video/mp4"})
    if image_url:
        content.append({"type": "input_file", "file_uri": image_url, "mime_type": "image/jpeg"})

    messages = []
    if system:
        messages.append({"role": "system", "content": system})
    messages.append({"role": "user", "content": content})

    body = {
        "model": model,
        "messages": messages,
        "temperature": temperature,
        "max_tokens": max_tokens,
    }
    if response_json:
        # Tells Gemini to bias toward JSON output. Other providers ignore this.
        body["extra_body"] = {"metadata": {"response_mime_type": "application/json"}}

    r = requests.post(f"{VLM_HOST}/vu/chat/completions", headers=HEADERS,
                      json=body, timeout=180)
    r.raise_for_status()
    envelope = r.json()
    if envelope.get("status") == "errored" or envelope.get("error"):
        err = envelope.get("error") or {}
        raise RuntimeError(f"VLM error: {err.get('code')} {err.get('message')}")
    choices = envelope.get("choices") or []
    if not choices:
        raise RuntimeError(f"VLM returned no choices: {envelope}")
    # Two shapes observed in the wild.
    text = choices[0].get("text") or (choices[0].get("message") or {}).get("content", "")
    if response_json:
        text = _strip_json_fence(text)
    return text


def _strip_json_fence(text):
    """Gemini often wraps JSON output in ```json ... ``` fences even when
    response_mime_type=application/json. Trim them so json.loads() works."""
    if not text:
        return text
    s = text.strip()
    if s.startswith("```"):
        s = s.split("\n", 1)[1] if "\n" in s else s[3:]
        if s.endswith("```"):
            s = s[:-3].rstrip()
    return s


In [ ]:
def resolve_media_url(video_no):
    """Bridge a videoNo to a public URL the VLM can fetch.

    `/download` streams the raw bytes back to you — it does NOT return a
    hosted URL. To feed a video to /vu/chat/completions you must host it
    yourself. This helper expects either:

      • MEMORIES_MEDIA_URL_TEMPLATE env var (e.g. https://your-cdn/{video_no}.mp4)
      • a MEDIA_URL_MAP dict you populate inline

    If neither is configured, raises so the notebook stops cleanly.
    """
    if video_no in MEDIA_URL_MAP:
        return MEDIA_URL_MAP[video_no]
    tpl = os.environ.get("MEMORIES_MEDIA_URL_TEMPLATE")
    if tpl:
        return tpl.format(video_no=video_no)
    raise RuntimeError(
        f"No media-URL bridge configured for {video_no}. "
        "Set MEMORIES_MEDIA_URL_TEMPLATE or add an entry to MEDIA_URL_MAP."
    )

# Per-notebook overrides: populate this for testing without a CDN.
# A public Memories.ai test asset is included as an example.
MEDIA_URL_MAP = {
    # "VI676024023022092288": "https://storage.googleapis.com/memories-test-data/test_1min.mp4",
}


## Step 1 — set the query

For demonstration we'll do a generic query — for a real lecture-library
Visual RAG, use queries like "every moment the professor writes an
equation on the whiteboard".


In [ ]:
QUERY        = "an object or person on a surface"
VISUAL_QUERY = QUERY                              # what the camera sees
AUDIO_QUERY  = QUERY                              # what someone says
EXTRACT_HINT = "one short descriptive label"      # what to pull out if we match
UNIQUE_ID    = "default"

seed_hits = search("a person", top_k=1, filtering_level=None)
VIDEO_NO  = seed_hits[0]["videoNo"]


## Step 2 — two-channel retrieval

Visual (`BY_CLIP`) finds frames that *look* like the query; audio
(`BY_AUDIO`) finds moments where someone *says* something matching the
query. We cast a wide net (`filtering_level="low"`) on both channels.


In [ ]:
visual_hits = search(VISUAL_QUERY, video_nos=[VIDEO_NO], top_k=10,
                     filtering_level="low", search_type="BY_CLIP")
audio_hits  = search(AUDIO_QUERY,  video_nos=[VIDEO_NO], top_k=10,
                     filtering_level="low", search_type="BY_AUDIO")

print(f"visual: {len(visual_hits)} hits, audio: {len(audio_hits)} hits")


## Step 3 — merge overlapping time-ranges across channels

If the visual and audio channels both fire at the same moment, that's
much stronger evidence than either alone. We merge time-ranges that
overlap (with ±2s slack) into a single candidate that records which
channels contributed.


In [ ]:
def to_event(hit, channel):
    return {
        "videoNo": hit["videoNo"],
        "start": float(hit["startTime"]),
        "end": float(hit["endTime"]),
        "score": float(hit["score"]),
        "channels": {channel},
        "audio_ts": hit.get("audio_ts"),
    }

def merge_overlapping(events, pad=2.0):
    events.sort(key=lambda e: (e["videoNo"], e["start"]))
    merged = []
    for e in events:
        if (merged and merged[-1]["videoNo"] == e["videoNo"]
                and e["start"] - pad <= merged[-1]["end"]):
            merged[-1]["end"] = max(merged[-1]["end"], e["end"])
            merged[-1]["channels"].update(e["channels"])
            merged[-1]["score"] = max(merged[-1]["score"], e["score"])
        else:
            merged.append({**e, "channels": set(e["channels"])})
    return [{**m, "channels": sorted(m["channels"])} for m in merged]

events = [to_event(h, "visual") for h in visual_hits] + \
         [to_event(h, "audio")  for h in audio_hits]
candidates = merge_overlapping(events)
print(f"\nmerged: {len(candidates)} candidates (from {len(events)} raw hits)")
for c in candidates[:5]:
    print(f"  {c['start']:>5.1f}-{c['end']:>5.1f}s  channels={c['channels']}  score={c['score']:.3f}")


## Step 4 — VLM verify each candidate

For each merged candidate, ask Gemini with a strict-JSON prompt:
does this clip actually match the query? If so, extract the requested
info. Keep candidates the VLM marks as `has_match=true`.


In [ ]:
MEDIA_URL_MAP[VIDEO_NO] = "https://storage.googleapis.com/memories-test-data/test_1min.mp4"
SYSTEM = ("You verify whether a candidate clip actually matches the user's "
          "query. Be strict: only has_match=true if visual or audio evidence "
          "is clear. Reply JSON only.")

verified = []
for i, c in enumerate(candidates[:5]):  # top 5 by merged score
    media_url = resolve_media_url(c["videoNo"])
    extract_clause = (f' If has_match is true, also fill "extract": {EXTRACT_HINT}.'
                      if EXTRACT_HINT else "")
    prompt = (
        f'Between {c["start"]:.0f}s and {c["end"]:.0f}s, does this clip match '
        f'the query: "{QUERY}"? '
        'Reply JSON: {"has_match": bool, "extract": str|null, '
        '"confidence": "low"|"medium"|"high"}.' + extract_clause
    )
    raw = vlm_complete(prompt, video_url=media_url, system=SYSTEM)
    try:
        verdict = json.loads(raw)
    except json.JSONDecodeError:
        verdict = {"raw": raw, "parse_error": True}
    print(f"[{i+1}] {c['start']:.0f}-{c['end']:.0f}s  channels={c['channels']}  -> {verdict}")
    if isinstance(verdict, dict) and verdict.get("has_match"):
        verified.append({"candidate": c, "verdict": verdict})

print(f"\n{len(verified)} of {min(5, len(candidates))} candidates verified as matches")


## Step 5 — synthesize the final answer with citations

Visual RAG's defining feature: every claim links to a specific
timestamp in a specific video. The user can verify any claim by
watching the cited evidence.


In [ ]:
if verified:
    print(f"Answer to: {QUERY!r}\n")
    for v in verified:
        c = v["candidate"]; verdict = v["verdict"]
        print(f"  • {verdict.get('extract')!r}  (confidence={verdict.get('confidence')})")
        print(f"    cited: video={c['videoNo']} @ {c['start']:.0f}-{c['end']:.0f}s")
        print(f"    channels={c['channels']}\n")
else:
    print("No verified matches.")
